# 📘 파일과 경로

파일 시스템 작업은 파이썬 프로그래밍에서 자주 필요합니다.
`pathlib`, `glob`, `shutil`, `zipfile`로 파일과 경로를 다뤄봅니다.

**학습 목표:**
- pathlib: 현대적 경로 처리
- glob: 패턴으로 파일 검색
- shutil: 파일/디렉토리 복사, 이동, 삭제
- zipfile: ZIP 아카이브 생성과 해제

## 1. pathlib — 현대적 경로 처리

`pathlib.Path`는 객체 지향적이고 직관적인 경로 처리를 제공합니다.

In [ ]:
from pathlib import Path
import tempfile

# ┌─────────────────────────────────────────┐
# │  pathlib.Path 주요 기능                   │
# │  Path() /   → 경로 결합                    │
# │  .exists()   → 경로 존재 확인               │
# │  .read_text()/.write_text() → 파일 읽기/쓰기│
# │  .name/.stem/.suffix → 경로 분석            │
# │  .parent/.parents → 상위 경로               │
# └─────────────────────────────────────────┘

tmp = Path(tempfile.gettempdir()) / "pydev_tutorial"
tmp.mkdir(parents=True, exist_ok=True)

# 파일 쓰기/읽기
hello = tmp / "hello.txt"
hello.write_text("안녕하세요! pathlib 튜토리얼입니다.", encoding="utf-8")
print(f"파일 내용: {hello.read_text(encoding='utf-8')}")

# 경로 분석
sample = Path("/home/user/documents/report.txt")
print(f"\n경로 분석:")
print(f"  전체: {sample}")
print(f"  파일명: {sample.name}")
print(f"  스템: {sample.stem}")
print(f"  확장자: {sample.suffix}")
print(f"  부모: {sample.parent}")

# 디렉토리 순회
print(f"\n튜토리얼 디렉토리 내용:")
for p in tmp.iterdir():
    print(f"  {p.name}")

# 정리
import shutil
shutil.rmtree(tmp, ignore_errors=True)

## 2. glob — 패턴으로 파일 검색

`glob`은 와일드카드 패턴으로 파일을 검색합니다.

In [ ]:
import glob
import os
from pathlib import Path

# ┌─────────────────────────────────────────┐
# │  glob 패턴                                │
# │  *      → 임의의 문자열                     │
# │  ?      → 임의의 1글자                     │
# │  [abc]  → a, b, c 중 하나                  │
# │  **/*   → 하위 디렉토리 포함 (recursive)   │
# └─────────────────────────────────────────┘

# 현재 디렉토리의 파이썬 파일
py_files = glob.glob("*.py")
print(f"현재 디렉토리 .py 파일: {len(py_files)}개")

# 특정 확장자
txt_files = glob.glob("*.txt")
print(f".txt 파일: {len(txt_files)}개")

# 중첩 디렉토리 검색
all_py = glob.glob("**/*.py", recursive=True)
print(f"\n하위 디렉토리 포함 .py 파일: {len(all_py)}개")

# iglob: 이터레이터 버전 (메모리 절약)
print("\niglob 결과 (처음 5개):")
for i, f in enumerate(glob.iglob("**/*.md", recursive=True)):
    if i >= 5:
        break
    print(f"  {f}")

# pathlib의 glob (더 현대적)
print("\npathlib.Path.glob:")
for p in Path(".").glob("*.md"):
    print(f"  {p}")

## 3. shutil — 파일/디렉토리 관리

`shutil`은 파일 복사, 이동, 삭제, 아카이브 등을 제공합니다.

In [ ]:
import shutil
import os
from pathlib import Path
import tempfile

# ┌─────────────────────────────────────────┐
# │  shutil 주요 기능                         │
# │  shutil.copy()     → 파일 복사 (메타데이터 X) │
# │  shutil.copy2()    → 파일 복사 (메타데이터 O) │
# │  shutil.copytree() → 디렉토리 복사           │
# │  shutil.move()     → 파일/디렉토리 이동      │
# │  shutil.rmtree()   → 디렉토리 삭제           │
# └─────────────────────────────────────────┘

tmp = Path(tempfile.mkdtemp(prefix="shutil_demo_"))
src = tmp / "source"
dst = tmp / "backup"
src.mkdir()

# 테스트 파일 생성
(src / "data.txt").write_text("중요한 데이터", encoding="utf-8")
(src / "config.json").write_text('{"key": "value"}', encoding="utf-8")
print(f"원본 파일: {os.listdir(src)}")

# 디렉토리 전체 복사
shutil.copytree(src, dst)
print(f"백업 파일: {os.listdir(dst)}")

# 단일 파일 복사
shutil.copy2(src / "data.txt", dst / "data_copy.txt")
print(f"복사 후: {os.listdir(dst)}")

# 파일 이동
moved = tmp / "moved.txt"
shutil.move(str(src / "data.txt"), str(moved))
print(f"이동 후 원본: {os.listdir(src)}")
print(f"이동된 파일 존재: {moved.exists()}")

# 디스크 사용량
total, used, free = shutil.disk_usage(tmp)
print(f"\n디스크: 전체 {total//1024//1024}MB, 사용 {used//1024//1024}MB, 여유 {free//1024//1024}MB")

# 정리
shutil.rmtree(tmp, ignore_errors=True)

## 4. zipfile — ZIP 아카이브

`zipfile`은 ZIP 파일의 생성, 읽기, 해제를 제공합니다.

In [ ]:
import zipfile
import tempfile
from pathlib import Path

# ┌─────────────────────────────────────────┐
# │  zipfile 주요 기능                        │
# │  ZipFile(name, 'w') → ZIP 생성            │
# │  ZipFile(name, 'r') → ZIP 읽기            │
# │  .write()          → 파일 추가              │
# │  .extractall()     → 전체 해제              │
# │  .namelist()       → 파일 목록              │
# └─────────────────────────────────────────┘

tmp = Path(tempfile.mkdtemp(prefix="zip_demo_"))

# 테스트 파일 생성
(tmp / "doc1.txt").write_text("문서 1", encoding="utf-8")
(tmp / "doc2.txt").write_text("문서 2", encoding="utf-8")

# ZIP 생성
zip_path = tmp / "archive.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(tmp / "doc1.txt", "doc1.txt")
    zf.write(tmp / "doc2.txt", "doc2.txt")
    zf.writestr("readme.txt", "ZIP 아카이브 안내서")

print(f"ZIP 생성: {zip_path}")

# ZIP 내용 확인
with zipfile.ZipFile(zip_path, 'r') as zf:
    print(f"\nZIP 내용:")
    for info in zf.infolist():
        print(f"  {info.filename}: {info.file_size}바이트")

# ZIP 해제
extract_dir = tmp / "extracted"
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)
print(f"\n해제된 파일: {os.listdir(extract_dir)}")

import os
# 정리
import shutil
shutil.rmtree(tmp, ignore_errors=True)

## 🎯 연습 문제

1. `pathlib.Path`를 사용해 파일을 생성하고, `.name`, `.stem`, `.suffix` 속성을 출력하세요.
2. `glob.glob()`를 사용해 현재 디렉토리의 모든 `.md` 파일을 찾으세요.
3. `shutil.copytree()`로 디렉토리를 백업하고 `shutil.rmtree()`로 삭제하는 코드를 작성하세요.
4. `zipfile`을 사용해 여러 텍스트 파일을 ZIP으로 압축하고 다시 해제하세요.